# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages, especially `vllm`, are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Make uv available in this shell even before restarting.
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies. LoRA packages are included so this notebook can train adapters too.
!.venv/bin/python -m pip install \
    sympy numpy tqdm bitsandbytes datasets peft trl accelerate \
    "vllm==0.8.5.post1" "transformers>=4.51,<4.54" "tokenizers>=0.21,<0.22" \
    antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel -> select another kernel -> Jupyter Kernel -> Python (cse151b).")


### Run the cell below every time to activate the installed environment. 

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import os
import random

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/frq_baseline_full.jsonl"
ERROR_REPORT_PATH = "results/frq_baseline_errors.jsonl"
MAX_TOKENS  = 4096
EVAL_FRQ_ONLY   = True
FRQ_SAMPLE_SIZE = None
GEN_BATCH_SIZE  = 5
SAMPLE_SEED     = 151
RUN_LIMIT       = None               # Optional extra cap after sampling

# ── Optional LoRA settings ────────────────────────────────────────────────────
# Keep these False for a normal baseline/submission run.
# Train in a fresh kernel before loading vLLM, then set USE_LORA=True to evaluate.
USE_LORA          = False
LORA_ADAPTER_PATH = "outputs/qwen-frq-lora"
TRAIN_LORA        = False
PREPARE_SFT_DATA  = False
SFT_INPUT_PATHS   = []               # Do NOT put data/public.jsonl here for real eval; use separate train/external data.
SFT_TRAIN_PATH    = "data/train_sft_frq.jsonl"
LORA_OUTPUT_DIR   = "outputs/qwen-frq-lora"
LORA_EPOCHS       = 1.0
LORA_MAX_LENGTH   = 4096
LORA_LR           = 2e-4
LORA_BATCH_SIZE   = 1
LORA_GRAD_ACCUM   = 16
LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.05

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_USE_V1"] = "0"  # More reliable inside notebooks

import re
import sys
from pathlib import Path
from typing import Any, Optional

from vllm import LLM, SamplingParams
from tqdm import tqdm


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = """You are solving free-response math problems for an automatic grader.

Each problem may contain one or more [ANS] blanks. Solve efficiently, then output the final answers in the exact format expected by the grader.

Keep your reasoning concise. Do not debate multiple strategies, do not repeat calculations, and do not explain basic definitions unless needed. After you have the answers, stop reasoning and write the boxed final answer immediately.

Rules:
1. End with exactly one boxed answer and write nothing after it.
   For one answer: \\boxed{answer}
   For multiple answers: \\boxed{answer1, answer2, answer3}
2. Put answers in the same order as the blanks or subquestions. If there is no [ANS] blank, infer the single requested final answer.
3. Use plain text math inside the box: ^ for powers, * for multiplication, same variable names as the problem, and functions like sqrt, sin, cos, tan, ln, log, atan.
4. Do not include units unless the problem explicitly requires units in the answer.
5. Do not use thousands separators, since commas separate multiple answers. Write 5850000, not 5,850,000.
6. Prefer exact expressions when natural. Keep clean fractions as reduced fractions, not decimals. Leave products unexpanded when asked.
7. For decimals, give about 12-15 significant digits unless the problem explicitly says to round. Obey nearest integer, cents, decimal-place, and significant-figure instructions exactly.
8. For embedded choice blanks, output only the requested letter or letters. If multiple letters are selected for one blank, concatenate them alphabetically with no spaces, e.g. CF.
9. For money, include $ only if the problem explicitly says the answer must begin with a dollar sign. For percent blanks, include % only when the problem explicitly asks for percent notation.
10. Before finalizing, check the answer count, order, signs, rounding, and formatting. Then output the boxed answer immediately.

Final response must end with exactly one line and no trailing explanation:
\\boxed{...}"""

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## Optional: Prepare FRQ SFT Data for LoRA

Run this only when you are training an adapter. Keep `data/public.jsonl` as eval-only for the real competition workflow; use your separate training set or external FRQ-style math data.


In [ ]:
def clean_question(question: str) -> str:
    question = question.replace("\r\n", "\n").replace("\r", "\n")
    question = "\n".join(line.rstrip() for line in question.splitlines())
    return question.strip()


def answer_text(answer) -> str:
    if isinstance(answer, list):
        return ", ".join(str(x).strip() for x in answer)
    return str(answer).strip()


def read_jsonl(path: str | Path) -> list[dict]:
    return [json.loads(line) for line in open(path) if line.strip()]


def write_jsonl(path: str | Path, rows: list[dict]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


def build_frq_sft_records(rows: list[dict], sample_size: int | None = None, seed: int = 151) -> list[dict]:
    rows = [row for row in rows if not row.get("options")]
    rng = random.Random(seed)
    rng.shuffle(rows)
    if sample_size is not None:
        rows = rows[:sample_size]

    records = []
    for row in rows:
        final_answer = answer_text(row["answer"])
        records.append({
            "id": row.get("id"),
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT_MATH},
                {"role": "user", "content": clean_question(row["question"])},
                {"role": "assistant", "content": f"\\boxed{{{final_answer}}}"},
            ],
        })
    return records


if PREPARE_SFT_DATA:
    if not SFT_INPUT_PATHS:
        raise ValueError("Set SFT_INPUT_PATHS to one or more train/external JSONL files before preparing SFT data.")
    train_rows = []
    for input_path in SFT_INPUT_PATHS:
        train_rows.extend(read_jsonl(input_path))
    sft_records = build_frq_sft_records(train_rows, sample_size=None, seed=SAMPLE_SEED)
    write_jsonl(SFT_TRAIN_PATH, sft_records)
    print(f"Wrote {len(sft_records)} SFT records to {SFT_TRAIN_PATH}")
else:
    print("Skipping SFT data prep. Set PREPARE_SFT_DATA=True when training LoRA.")


## Optional: Train QLoRA Adapter

Run this in a fresh kernel before loading vLLM. After training, restart/clear GPU memory, set `USE_LORA=True`, and run the normal vLLM eval cells.


In [ ]:
if TRAIN_LORA:
    import torch
    from datasets import load_dataset
    from peft import LoraConfig, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from trl import SFTConfig, SFTTrainer

    tokenizer_train = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer_train.pad_token is None:
        tokenizer_train.pad_token = tokenizer_train.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model_train = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model_train = prepare_model_for_kbit_training(model_train)

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    )

    dataset_train = load_dataset("json", data_files=SFT_TRAIN_PATH, split="train")

    def formatting_func(example):
        return tokenizer_train.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )

    training_args = SFTConfig(
        output_dir=LORA_OUTPUT_DIR,
        per_device_train_batch_size=LORA_BATCH_SIZE,
        gradient_accumulation_steps=LORA_GRAD_ACCUM,
        learning_rate=LORA_LR,
        num_train_epochs=LORA_EPOCHS,
        max_length=LORA_MAX_LENGTH,
        bf16=True,
        logging_steps=10,
        save_steps=250,
        save_total_limit=2,
        packing=False,
        report_to=[],
    )

    trainer = SFTTrainer(
        model=model_train,
        args=training_args,
        train_dataset=dataset_train,
        peft_config=peft_config,
        formatting_func=formatting_func,
    )
    trainer.train()
    trainer.save_model(LORA_OUTPUT_DIR)
    tokenizer_train.save_pretrained(LORA_OUTPUT_DIR)
    print(f"Saved LoRA adapter to {LORA_OUTPUT_DIR}")

    # Free memory before loading vLLM in this same kernel.
    del trainer, model_train
    torch.cuda.empty_cache()
else:
    print("Skipping LoRA training. Set TRAIN_LORA=True only when you want to train an adapter.")


## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **BitsAndBytes quantization** through vLLM.  
This keeps inference fast and memory-efficient on a single 48GB A40.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel
- `max_num_batched_tokens` — total prompt/generation tokens vLLM may batch at once

In [ ]:
llm_kwargs = dict(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.78,
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=4,
    max_num_batched_tokens=8192,
)
if USE_LORA:
    llm_kwargs["enable_lora"] = True

llm = LLM(**llm_kwargs)

tokenizer = llm.get_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

lora_request = None
if USE_LORA:
    from vllm.lora.request import LoRARequest
    if not Path(LORA_ADAPTER_PATH).exists():
        raise FileNotFoundError(f"LoRA adapter not found: {LORA_ADAPTER_PATH}")
    lora_request = LoRARequest("frq_lora", 1, LORA_ADAPTER_PATH)

print("vLLM model loaded.", "LoRA enabled." if USE_LORA else "Baseline model.")


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [ ]:
# Build prompts
eval_pool = [item for item in data if not item.get("options")] if EVAL_FRQ_ONLY else data
if FRQ_SAMPLE_SIZE is not None and EVAL_FRQ_ONLY:
    rng = random.Random(SAMPLE_SEED)
    eval_data = rng.sample(eval_pool, min(FRQ_SAMPLE_SIZE, len(eval_pool)))
else:
    eval_data = eval_pool
if RUN_LIMIT is not None:
    eval_data = eval_data[:RUN_LIMIT]

print(f"Evaluation set: {len(eval_data)} questions ({sum(bool(d.get('options')) for d in eval_data)} MCQ, {sum(not d.get('options') for d in eval_data)} free-form)")
prompts = []
for item in eval_data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate with vLLM in chunks to keep GPU memory stable
print(f"Generating responses for {len(prompts)} questions in batches of {GEN_BATCH_SIZE}...")
responses = []
for start in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc="Generating"):
    batch_prompts = prompts[start:start + GEN_BATCH_SIZE]
    batch_outputs = llm.generate(batch_prompts, sampling_params=sampling_params, lora_request=lora_request)
    responses.extend(out.outputs[0].text.strip() for out in batch_outputs)

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={eval_data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# ── Inlined FRQ postprocess helpers. This keeps the notebook self-contained. ──
UNIT_WORDS = (
    "degrees fahrenheit", "degrees celsius", "degrees kelvin", "degrees rankine",
    "fahrenheit", "celsius", "kelvin", "rankine", "degrees", "degree",
    "hours", "hour", "minutes", "minute", "seconds", "second", "years", "year",
    "feet", "foot", "meters", "meter", "miles", "mile", "dollars", "dollar",
)


def _find_boxed_entries(text: str) -> list[tuple[int, int, str]]:
    entries = []
    start = 0
    while True:
        idx = text.find("\\boxed{", start)
        if idx < 0:
            break
        brace_start = idx + len("\\boxed{")
        depth = 1
        i = brace_start
        while i < len(text) and depth > 0:
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
            i += 1
        if depth == 0:
            entries.append((idx, i, text[brace_start:i - 1].strip()))
        start = max(i, idx + 1)
    return entries


def extract_final_answer_text(response: str) -> tuple[str, list[str]]:
    notes = []
    text = response.strip()
    think_end = text.rfind("</think>")
    if think_end >= 0:
        text = text[think_end + len("</think>"):].strip()
        notes.append("removed_think_prefix")

    entries = _find_boxed_entries(text)
    if entries:
        last_group = [entries[-1]]
        for i in range(len(entries) - 2, -1, -1):
            gap = text[entries[i][1]:entries[i + 1][0]]
            if re.fullmatch(r"[\s,\$.;:\-&\\]*", gap or ""):
                last_group.insert(0, entries[i])
            else:
                break
        notes.append("extracted_boxed")
        return ", ".join(entry[2] for entry in last_group), notes

    marker_patterns = [r"FINAL\s*:", r"Final answer\s*:", r"final answer\s*:", r"Answer\s*:", r"answer\s*:", r"answer is", r"####", r"# Answer"]
    for pattern in marker_patterns:
        matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))
        if matches:
            notes.append("extracted_marker")
            return text[matches[-1].end():].strip(), notes

    notes.append("no_explicit_answer")
    return text, notes


def _question_requires_dollar(question: str) -> bool:
    q = question.lower()
    return "must begin with a dollar sign" in q or "must begin with $" in q


def _question_requires_percent(question: str) -> bool:
    q = question.lower()
    return "fill in the blank with a percent" in q or "include %" in q


def _remove_thousands_commas(text: str) -> str:
    return re.sub(r"(?<!\.\d)(?<=\d),(?=\d{3}(?!\.\d)(?:\D|$))", "", text)


def _strip_outer_box_or_dollars(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^\\boxed\{(.*)\}$", r"\1", text)
    if len(text) >= 2 and text[0] == "$" and text[-1] == "$":
        text = text[1:-1].strip()
    return text


def _read_braced(text: str, brace_idx: int) -> tuple[str, int] | None:
    if brace_idx >= len(text) or text[brace_idx] != "{":
        return None
    depth = 1
    i = brace_idx + 1
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    if depth != 0:
        return None
    return text[brace_idx + 1:i - 1], i


def _latex_frac_to_plain(text: str) -> str:
    for command in ("\\dfrac", "\\frac"):
        start = 0
        while True:
            idx = text.find(command + "{", start)
            if idx < 0:
                break
            first_start = idx + len(command)
            first = _read_braced(text, first_start)
            if first is None:
                start = idx + 1
                continue
            numerator, first_end = first
            second = _read_braced(text, first_end)
            if second is None:
                start = first_end
                continue
            denominator, second_end = second
            text = text[:idx] + f"({numerator})/({denominator})" + text[second_end:]
            start = idx + 1
    return text


def _normalize_latex_math(text: str) -> str:
    text = text.replace("\\left", "").replace("\\right", "")
    text = text.replace("\\,", "")
    text = text.replace("\\cdot", "*").replace("\\times", "*")
    text = text.replace("\\pi", "pi")
    text = text.replace("\\infty", "infinity")
    text = text.replace("\\ln", "ln")
    text = text.replace("\\log", "log")
    text = text.replace("\\sin", "sin")
    text = text.replace("\\cos", "cos")
    text = text.replace("\\tan", "tan")
    text = text.replace("\\sqrt", "sqrt")
    text = _latex_frac_to_plain(text)
    text = re.sub(r"([A-Za-z0-9_)])\^\{([^{}]+)\}", r"\1^(\2)", text)
    text = re.sub(r"\be\^\(?([^),\s]+)\)?", r"e^(\1)", text)
    text = re.sub(r"(?<=\))(?=\()", "*", text)
    text = re.sub(r"(?<=\d)(?=[A-Za-z(])", "*", text)
    text = re.sub(r"(?<=[A-Za-z)])(?=\d)", "*", text)
    return text


def _normalize_separators(text: str, expected_count: int) -> str:
    text = text.replace("|||", ",")
    if expected_count > 1:
        text = re.sub(r"[\n;]+", ",", text)
    else:
        text = text.replace("\n", " ")
    text = re.sub(r"\s*,\s*", ", ", text)
    text = re.sub(r",\s*,+", ", ", text)
    return text.strip(" ,.")


def _split_top_level_commas(text: str) -> list[str]:
    parts = []
    depth = 0
    start = 0
    pairs = {"(": ")", "[": "]", "{": "}", "<": ">"}
    closers = set(pairs.values())
    for i, char in enumerate(text):
        if char in pairs:
            depth += 1
        elif char in closers and depth > 0:
            depth -= 1
        elif char == "," and depth == 0:
            parts.append(text[start:i].strip())
            start = i + 1
    parts.append(text[start:].strip())
    return parts


def _strip_units_from_part(part: str, question: str) -> str:
    preserve_symbols = _question_requires_dollar(question) or _question_requires_percent(question)
    out = part.strip().replace("\\%", "%")
    if not _question_requires_dollar(question):
        out = re.sub(r"^\$\s*", "", out)
    if not _question_requires_percent(question):
        out = re.sub(r"\s*percent$", "", out, flags=re.IGNORECASE)
    if not preserve_symbols:
        out = re.sub(r"\s+(?:%s)\.?$" % "|".join(re.escape(u) for u in UNIT_WORDS), "", out, flags=re.IGNORECASE)
    return out.strip()


def postprocess_response(response: str, question: str, expected_count: int) -> dict[str, Any]:
    answer_text, notes = extract_final_answer_text(response)
    answer_text = _strip_outer_box_or_dollars(answer_text)
    normalized = _normalize_latex_math(answer_text)
    if normalized != answer_text:
        answer_text = normalized
        notes.append("normalized_latex_math")
    answer_text = _remove_thousands_commas(answer_text)
    answer_text = _normalize_separators(answer_text, expected_count)
    parts = _split_top_level_commas(answer_text) if expected_count > 1 else [answer_text.strip()]
    parts = [_strip_units_from_part(part, question) for part in parts]
    answer_text = ", ".join(part for part in parts if part != "")
    return {"answer_text": answer_text, "response": f"\\boxed{{{answer_text}}}", "notes": notes}


def _gold_list(gold: Any) -> list[str]:
    return [str(x) for x in gold] if isinstance(gold, list) else [str(gold)]


def _safe_judge(judger: Any, pred: str, gold: Any) -> bool:
    gold_items = _gold_list(gold)
    try:
        return bool(judger.auto_judge(pred=pred, gold=gold_items, options=[[]] * len(gold_items)))
    except Exception:
        return False


def _answer_count_error(answer_text: str, gold: Any) -> bool:
    return len(_split_top_level_commas(answer_text)) != len(_gold_list(gold))


def classify_frq_error(judger: Any, item: dict, response: str, post: dict[str, Any]) -> str:
    gold = item["answer"]
    if _answer_count_error(post["answer_text"], gold):
        return "wrong_answer_count"
    if "no_explicit_answer" in post["notes"]:
        return "missing_boxed"
    if any(char in post["answer_text"] for char in ["\\", "{", "}"]):
        return "expression_format"
    if re.search(r"\d+\.\d{1,3}$", post["answer_text"]) and any("." in g and len(g.split(".")[-1]) > 4 for g in _gold_list(gold)):
        return "precision_rounding"
    return "wrong_math"


def score_frq_item(judger: Any, item: dict, response: str) -> dict[str, Any]:
    gold = item["answer"]
    expected_count = len(_gold_list(gold))
    raw_correct = _safe_judge(judger, response, gold)
    post = postprocess_response(response, item["question"], expected_count)
    post_correct = _safe_judge(judger, post["response"], gold)

    if not post_correct:
        gold_items = _gold_list(gold)
        if all(g.isupper() for g in gold_items):
            upper = post["answer_text"].upper()
            upper_response = f"\\boxed{{{upper}}}"
            if _safe_judge(judger, upper_response, gold):
                post["answer_text"] = upper
                post["response"] = upper_response
                post["notes"].append("uppercased_answer")
                post_correct = True

    final_correct = raw_correct or post_correct
    if final_correct:
        error_type = "correct" if raw_correct else "correct_after_postprocess"
    else:
        error_type = classify_frq_error(judger, item, response, post)

    return {
        "raw_correct": raw_correct,
        "postprocess_correct": post_correct,
        "correct": final_correct,
        "postprocessed_response": post["response"],
        "postprocessed_answer": post["answer_text"],
        "postprocess_notes": post["notes"],
        "error_type": error_type,
    }


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(eval_data, responses), total=len(eval_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
        score_info = {
            "raw_correct": correct,
            "postprocess_correct": correct,
            "postprocessed_response": response,
            "postprocessed_answer": extract_letter(response),
            "postprocess_notes": [],
            "error_type": "correct" if correct else "wrong_math",
        }
    else:
        score_info = score_frq_item(judger, item, response)
        correct = score_info["correct"]

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "postprocessed_response": score_info["postprocessed_response"],
        "postprocessed_answer": score_info["postprocessed_answer"],
        "postprocess_notes": score_info["postprocess_notes"],
        "raw_correct": score_info["raw_correct"],
        "postprocess_correct": score_info["postprocess_correct"],
        "correct":  correct,
        "error_type": score_info["error_type"],
    })

print(f"Scoring complete. {len(results)} results.")


## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  Raw FRQ    : {sum(r['raw_correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc([dict(r, correct=r['raw_correct']) for r in free_res]):.2f}%)")
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("\nError types:")
for error_type, count in sorted(__import__('collections').Counter(r['error_type'] for r in results).items()):
    print(f"  {error_type:24s} {count:4d}")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
error_path = Path(ERROR_REPORT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = r
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

wrong_results = [r for r in results if not r["correct"]]
with open(error_path, "w") as f:
    for r in wrong_results:
        f.write(json.dumps(r) + "\n")

print(f"Saved {len(results)} records to {out_path}")
print(f"Saved {len(wrong_results)} error records to {error_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!